# Infosys India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** career.infosys.com

**ATS Detection:** Automatic API + Selenium fallback

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml playwright -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path

# Add scripts dir to path so we can import scraper_utils
SCRIPTS_DIR = Path.home() / "Job_Scrapers" / "All_Scripts"
sys.path.insert(0, str(SCRIPTS_DIR))

from scraper_utils import *
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

print("Imports loaded. Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 23:38:05


In [3]:
COMPANY = "Infosys"
OUTPUT_DIR = get_output_dir(COMPANY)
print(f"Output directory: {OUTPUT_DIR}")


Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Infosys/Outputs/2026_03_31


In [4]:
print("=" * 60)
print("INFOSYS INDIA JOB SCRAPER")
print("Source: career.infosys.com (Angular SPA)")
print("=" * 60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


def fetch_jd_selenium(driver, url, timeout=10):
    """Visit a job detail page and extract the JD text."""
    try:
        driver.get(url)
        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, "lxml")
        # Try common JD container selectors
        for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                     "[class*='details']", "article", "main", ".content"]:
            el = soup.select_one(sel)
            if el and len(el.get_text(strip=True)) > 100:
                return el.get_text(" ", strip=True)
        # Fallback: get body text
        body = soup.select_one("body")
        return body.get_text(" ", strip=True)[:5000] if body else ""
    except Exception as e:
        print(f"    [WARN] JD fetch failed for {url}: {e}")
        return ""

def fetch_jd_requests(session, url):
    """Fetch a job detail page via requests and extract JD text."""
    try:
        resp = session.get(url, timeout=20)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml")
            for sel in ["[class*='job-description']", "[class*='jd-info']", "[class*='description']",
                         "[class*='details']", "article", "main"]:
                el = soup.select_one(sel)
                if el and len(el.get_text(strip=True)) > 100:
                    return el.get_text(" ", strip=True)
            body = soup.select_one("body")
            return body.get_text(" ", strip=True)[:5000] if body else ""
    except:
        pass
    return ""


infosys_jobs = []

# Strategy 1: Try intercepting the XHR API that the Angular SPA calls
session = get_session()
try:
    # Common Infosys career API patterns
    api_urls = [
        "https://career.infosys.com/api/joblist?location=India&limit=100",
        "https://career.infosys.com/api/jobs?country=India&limit=100",
    ]
    for api_url in api_urls:
        try:
            resp = session.get(api_url, timeout=15)
            if resp.status_code == 200:
                data = resp.json()
                jobs_list = data if isinstance(data, list) else data.get("jobs", data.get("data", data.get("results", [])))
                if jobs_list and len(jobs_list) > 0:
                    print(f"  API found at {api_url}: {len(jobs_list)} jobs")
                    for j in jobs_list:
                        title = j.get("title", j.get("jobTitle", j.get("name", "")))
                        if is_valid_job_title(title):
                            ref = j.get("jobReferenceCode", j.get("id", j.get("reqId", "")))
                            infosys_jobs.append({
                                "job_id": str(ref),
                                "title": title,
                                "company_name": "Infosys",
                                "raw_jd_text": j.get("description", j.get("jd", "")),
                                "location_city": j.get("location", j.get("city", "India")),
                                "industry": "IT Services",
                                "date_posted": j.get("postedDate", datetime.now().strftime("%Y-%m-%d"))[:10],
                                "is_active": True,
                                "job_url": f"https://career.infosys.com/jobdesc?jobReferenceCode={ref}" if ref else "",
                                "business_unit": j.get("department", j.get("category", "")),
                                "source_platform": "Infosys API",
                            })
                    break
        except:
            continue
except Exception as e:
    print(f"  API discovery: {e}")

# Strategy 2: Selenium with careful element targeting
if len(infosys_jobs) < 5:
    print("  Using Selenium on career.infosys.com/joblist...")
    driver = setup_selenium()
    try:
        driver.get("https://career.infosys.com/joblist")
        time.sleep(12)

        # Wait for Angular to render
        try:
            WebDriverWait(driver, 25).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='jobdesc'], a[href*='jobReferenceCode']"))
            )
        except:
            time.sleep(8)

        for page in range(10):
            soup = BeautifulSoup(driver.page_source, "lxml")

            # Remove nav/header/footer to avoid grabbing UI text
            for unwanted in soup.select("nav, header, footer, [role='navigation'], [class*='menu'], [class*='filter'], [class*='pagination-text']"):
                unwanted.decompose()

            # Primary: find links to job description pages
            job_links = soup.select("a[href*='jobdesc'], a[href*='jobReferenceCode']")

            new_count = 0
            for link in job_links:
                title = link.get_text(strip=True)
                href = link.get("href", "")

                # Extract ref code from URL
                ref_match = re.search(r"jobReferenceCode=([\w-]+)", href)
                ref_code = ref_match.group(1) if ref_match else href.split("/")[-1]

                # Get parent card for location
                card = link.parent
                if card:
                    loc_el = card.select_one("[class*='location'], [class*='city'], span")
                    loc = loc_el.get_text(strip=True) if loc_el else "India"
                else:
                    loc = "India"

                if is_valid_job_title(title) and title not in [j["title"] for j in infosys_jobs]:
                    full_url = href if href.startswith("http") else f"https://career.infosys.com{href}" if href else ""
                    infosys_jobs.append({
                        "job_id": ref_code,
                        "title": title,
                        "company_name": "Infosys",
                        "raw_jd_text": "",
                        "location_city": loc.split(",")[0].strip() if loc else "India",
                        "industry": "IT Services",
                        "date_posted": datetime.now().strftime("%Y-%m-%d"),
                        "is_active": True,
                        "job_url": full_url,
                        "business_unit": "",
                        "source_platform": "Infosys Selenium",
                    })
                    new_count += 1

            print(f"  Page {page+1}: {new_count} new jobs (total: {len(infosys_jobs)})")
            if new_count == 0 and page > 0:
                break

            # Try next page
            try:
                next_btn = driver.find_element(By.CSS_SELECTOR, "a[aria-label*='Next'], a[aria-label*='next'], button[aria-label*='Next']")
                driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(5)
            except:
                break

        # Fetch JDs for found jobs
        if infosys_jobs:
            print(f"\n  Fetching JD details for up to 40 jobs...")
            for i, job in enumerate(infosys_jobs[:40]):
                if job.get("raw_jd_text") and len(job["raw_jd_text"]) > 100:
                    continue
                if job["job_url"]:
                    jd = fetch_jd_selenium(driver, job["job_url"])
                    if jd:
                        infosys_jobs[i]["raw_jd_text"] = jd
                if (i + 1) % 10 == 0:
                    print(f"    Fetched {i+1}/{min(40, len(infosys_jobs))} JDs")

    except Exception as e:
        print(f"  Selenium error: {e}")
        import traceback; traceback.print_exc()
    finally:
        driver.quit()

print(f"Total Infosys India jobs: {len(infosys_jobs)}")


INFOSYS INDIA JOB SCRAPER
Source: career.infosys.com (Angular SPA)


  Using Selenium on career.infosys.com/joblist...


  Page 1: 0 new jobs (total: 0)


  Page 2: 0 new jobs (total: 0)
Total Infosys India jobs: 0


In [5]:
df_infosys = save_results(infosys_jobs, "Infosys", OUTPUT_DIR)
if df_infosys is not None:
    print(f"\nSample jobs:")
    cols = ["title","location_city","seniority_level","business_unit","job_url"]
    cols = [c for c in cols if c in df_infosys.columns]
    print(df_infosys[cols].head(10).to_string())


  [WARN] No jobs found for Infosys
